# DDR grading test set — one-time restricted evaluation of the frozen P and P-EP graders (record §82, §83)

**Evaluation only. Nothing is trained, tuned or selected on DDR.** Everything that defines the run is in
`ddr_grading_protocol_v1.json`: the two image lists by hash, the twelve checkpoint hashes, the one primary contrast, the
bootstrap, the source-family and same-visit analyses, the stop conditions. The image path, inference settings and parity
tolerances are those locked earlier (Stage 2 once, direct resize to 512, T4 / mixed_float16 / batch 8, CORN decode at 0.5).

- **Primary:** QWK(P-EP) − QWK(P), mean of the three seed-labelled pairs, on **3,540** images. Supported only if the paired
  95 % bootstrap interval is wholly above zero. The three differences are always shown; a non-positive one is flagged.
- **Secondary, descriptive:** E1-EP and E2-EP against P-EP; per-grade and per-cut metrics; calibration; seed ensemble.
- **Sensitivity only:** the broader 3,725 images. It contains 185 images that were processed in the lesion-probe
  experiments, so it is not an untouched set and never replaces the primary result.
- No IDRiD file is opened by this notebook.

Order of the cells — it cannot be changed:
1. Setup.
2. **Stage and preflight** (no model, no image decoded): the image archive and audit files are copied from the private Drive
   folder; protocols, twelve checkpoints, both lists, every image's SHA-256, the official size / CRC-32 comparison, the
   near-duplicate screen, and whether anything of this evaluation already exists.
3. **Parity gate**, twelve models: ten raw APTOS images through the image path and all 730 APTOS validation images against
   each model's stored table. No DDR image is read. If it fails, stop.
4. **The DDR run, once.** Refuses without the confirmation token, without a passed gate from this runtime and commit, or if
   a lock or run directory already exists.
5. Print the primary report.

GPU runtime (T4). **Limits that travel with the result:** file-name family is tied to grade in this test set (every grade-0
image is timestamp-named, every grade-4 image `007`-named), so the pooled QWK cannot separate grading from recognising the
source; DDR has no patient identifiers, so image-level intervals may be too narrow; the three P-EP checkpoints share one
adapted encoder; the mirror agrees with the official archive in size and CRC-32, which is not proof of byte identity; a clean
near-duplicate screen does not prove independence from the development data; nothing here establishes clinical utility.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / Stage-4 weights needed; no IDRiD file is used.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad
import shutil, tarfile
import ddr_grading_data as dd, ddr_grading_eval as ev
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] STAGE AND PREFLIGHT (no model, no prediction, no image decoded) ====
PROTOCOL, PROTOCOL_SHA, PARENT = ev.load_protocol()
DRIVE_ROOT = colab_config.DRIVE.project_root
EXP = colab_config.DRIVE.experiments_root
OUT_ROOT = posixpath.join(EXP, "DDRGrading")
SOURCE = posixpath.join(colab_config.DATASET_ROOT, "DDR", "grading_test_v1")      # private Drive folder
LOCAL = "/content/ddr_grading_test_v1"
IMAGES, AUDIT = posixpath.join(LOCAL, "images"), posixpath.join(LOCAL, "audit")
N_IMAGES = PROTOCOL["populations"]["broader"]["n"]
if not os.path.isdir(IMAGES) or len(os.listdir(IMAGES)) != N_IMAGES:
    os.makedirs(IMAGES, exist_ok=True)
    _tar = posixpath.join(LOCAL, "images.tar")
    shutil.copyfile(posixpath.join(SOURCE, "ddr_grading_test_3725.tar"), _tar)
    with tarfile.open(_tar) as _archive:
        _members = _archive.getmembers()
        assert all(m.isfile() and "/" not in m.name and "\\" not in m.name and m.name.endswith(".jpg") for m in _members), "unexpected archive member"
        _archive.extractall(IMAGES)
    os.remove(_tar)
shutil.copytree(posixpath.join(SOURCE, "audit"), AUDIT, dirs_exist_ok=True)
assert len(os.listdir(IMAGES)) == N_IMAGES, "the staged folder does not hold exactly the listed images"
PREFLIGHT = ev.preflight(DRIVE_ROOT, IMAGES, AUDIT, OUT_ROOT)      # raises on the first mismatch; hashes every image
print("protocol", PROTOCOL_SHA, "| commit", REPO_COMMIT)
for _name, _entry in PROTOCOL["checkpoints"].items():
    print(f"  {_name:<15} {_entry['model_type']:<3} {_entry['sha256']}  {_entry['best_slot']} (epoch index {_entry['best_epoch_index']})")
print("images", PREFLIGHT["images"], "| primary", PREFLIGHT["primary_images"], PREFLIGHT["primary_grade_counts"],
      "| screen flagged", PREFLIGHT["screen_flagged"], "| inference:", PARENT["inference"])
print("screen", PREFLIGHT["screen_sha256"], "| official size / CRC-32 check", PREFLIGHT["official_check_sha256"])
print("PREFLIGHT PASS | existing outputs of this evaluation:", PREFLIGHT["existing_outputs"])

In [ ]:
# ==== [3] PARITY GATE (twelve models) -- must pass before any DDR image is decoded ====
GEN4 = cache.stage4_generation_id(PARENT["checkpoints"]["stage4_unet"]["sha256"], len(v2cfg.STAGE4_V2A_CLASSES))
BUNDLE_ID = cache.bundle_id(v2cfg.STAGE2_RGB_GENERATION, v2cfg.STAGE3_GENERATION, GEN4)
BUNDLE = ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256,
                        expected_stage4_sha256=PARENT["checkpoints"]["stage4_unet"]["sha256"])   # cached APTOS RGB frames, on Drive
GATE = ev.gates(DRIVE_ROOT, posixpath.join(colab_config.APTOS2019_RAW_DIR, "train_images"), BUNDLE,
                posixpath.join(OUT_ROOT, f"parity_gate_{PROTOCOL_SHA[:12]}"),
                aptos_processed_dir=posixpath.join(colab_config.APTOS2019_PROCESSED_DIR, "train_images"))
print(json.dumps({"PASS": GATE["PASS"], "failures": GATE["failures"],
                  "grading_output_dtype": GATE["checks"]["grading_output_dtype"],
                  "raw_subset_images": GATE["checks"]["raw_subset"]["image_checks"],
                  "raw_subset_predictions": GATE["checks"]["raw_subset"]["predictions"],
                  "full_validation_730": GATE["checks"]["full_validation"],
                  "ddr_images_read": GATE["ddr_images_read"], "environment": GATE["environment"]}, indent=1, default=float))
assert GATE["PASS"], "PARITY GATE FAILED -- stop here. Do not run the DDR cell. Report the failures."


In [ ]:
# ==== [4] THE ONE-TIME DDR RUN (P, P-EP, E1-EP, E2-EP on the 3,725 listed images) ====
# Runs once, and only after explicit authorisation. To run it, set CONFIRM to the token printed below. A second run is
# refused; after a technical failure only, set TECHNICAL_RERUN_REASON to a written reason (it is recorded with the run).
# Nothing is changed after seeing a result.
print("token:", PROTOCOL["confirmation_token"])
CONFIRM = None
TECHNICAL_RERUN_REASON = None
RESULT = ev.run(DRIVE_ROOT, IMAGES, AUDIT, OUT_ROOT, GATE, confirm=CONFIRM, technical_rerun_reason=TECHNICAL_RERUN_REASON,
                command="colab/notebooks/ddr_grading_v1.ipynb cell 4")

In [ ]:
# ==== [5] REPORT (reads what cell 4 wrote) ====
print(open(posixpath.join(RESULT["out_dir"], "primary_report.md"), encoding="utf-8").read())
print(json.dumps(json.load(open(posixpath.join(RESULT["out_dir"], "output_hashes.json"))), indent=1))
print("output:", RESULT["out_dir"])